# Notebook 03 – Integration of ISTAT data

**1. Load the two clean tables** 
* Accidents (one row per municipality and year, 2001–2024) and SITUAS (one row per municipality, population and area at 31/12/2024). SITUAS is read with `keep_default_na=False` so that "NA" (Napoli) and "None" (a town) are not turned into missing values again.

In [1]:
import pandas as pd
import os

df_accidents = pd.read_csv("Data/Processed/accidents_clean.csv")
df_situas = pd.read_csv("Data/Processed/situas_clean.csv", keep_default_na=False, na_values=[""])  

print("Accidents:", df_accidents.shape)                                        
print("SITUAS:", df_situas.shape)                                              
print("Sigla NA kept:", (df_situas["Sigla automobilistica"] == "NA").sum())   
df_accidents.head(3)

Accidents: (191184, 5)
SITUAS: (7896, 8)
Sigla NA kept: 92


,REF_AREA,TIME_PERIOD,accidents,deaths,injured
0,1001,2001,5,0,10
1,1001,2002,5,0,10
2,1001,2003,4,0,7


**2. Comparing the municipality codes** 
* The accidents table has 8,578 distinct codes (2001–2024); SITUAS has 7,896 (municipalities at 31/12/2024). The 682 codes only in the accidents table are municipalities that were merged or abolished before 2024: they have no population or area in SITUAS and will be excluded by the merge. No code is only in SITUAS: every 2024 municipality appears at least once in the accidents table.

In [2]:

print(df_accidents["TIME_PERIOD"].min(), df_accidents["TIME_PERIOD"].max())    

print("Accidents codes:", df_accidents["REF_AREA"].nunique())                  
print("SITUAS codes:", df_situas["Codice Comune (numerico)"].nunique())        

only_accidents = set(df_accidents["REF_AREA"].unique()) - set(df_situas["Codice Comune (numerico)"].unique())
print("Codes only in accidents:", len(only_accidents))

only_situas = set(df_situas["Codice Comune (numerico)"].unique()) - set(df_accidents["REF_AREA"].unique())
print("Codes only in SITUAS:", len(only_situas))

2001 2024
Accidents codes: 8578
SITUAS codes: 7896
Codes only in accidents: 682
Codes only in SITUAS: 0


**3. Merge and add zero-accident municipalities** 
* Left join between accidents and SITUAS on the municipality code.
* 2024 lists only municipalities with at least 1 accident: the 1,557 municipalities with 0 accidents are not included, so the 2024 average rate is slightly overestimated.

In [17]:
df = df_accidents.merge(
    df_situas,
    left_on="REF_AREA",
    right_on="Codice Comune (numerico)",
    how="left",
)
print("Rows after merge:", len(df))
print("Matched rows:", df["Comune"].notna().sum())
print("Unmatched rows:", df["Comune"].isna().sum())
df.head(3)

Rows after merge: 191184
Matched rows: 182569
Unmatched rows: 8615


,REF_AREA,TIME_PERIOD,accidents,deaths,injured,Codice Comune (numerico),Comune,Sigla automobilistica,Codice Regione,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione residente,Superficie (Kmq)
0,1001,2001,5,0,10,1001.0,Agliè,TO,1.0,0.0,0.0,2585.0,13.1463
1,1001,2002,5,0,10,1001.0,Agliè,TO,1.0,0.0,0.0,2585.0,13.1463
2,1001,2003,4,0,7,1001.0,Agliè,TO,1.0,0.0,0.0,2585.0,13.1463


In [5]:
matched = df[df["Comune"].notna()].groupby("TIME_PERIOD")["REF_AREA"].count().rename("matched")
unmatched = df[df["Comune"].isna()].groupby("TIME_PERIOD")["REF_AREA"].count().rename("unmatched")

by_year = pd.DataFrame({"matched": matched, "unmatched": unmatched}).fillna(0).astype(int).reset_index()
by_year["total"] = by_year["matched"] + by_year["unmatched"]
by_year

,TIME_PERIOD,matched,unmatched,total
0,2001,7534,567,8101
1,2002,7535,567,8102
2,2003,7535,567,8102
3,2004,7535,565,8100
4,2005,7536,565,8101
5,2006,7499,602,8101
6,2007,7499,602,8101
7,2008,7499,602,8101
8,2009,7500,600,8100
9,2010,7614,480,8094


In [6]:
print(by_year["unmatched"].sum())  

8615


- 682 municipality codes in the accidents table (2001–2024) have no match in SITUAS 2024, because those municipalities were merged or abolished. They account for 1.85% of total accidents (88,281 out of 4,779,141). They are kept in the dataset but excluded from the per-capita and per-km² rates, which require population and area data.

In [7]:
unmatched = (df[df["Comune"].isna()].groupby("TIME_PERIOD")["accidents"] .sum().reset_index(name="accidents"))
print("Distinct unmatched codes:", df.loc[df["Comune"].isna(), "REF_AREA"].nunique())
print("Unmatched accidents:", unmatched["accidents"].sum())
print("Share:", round(unmatched["accidents"].sum() / df["accidents"].sum() * 100, 2), "%")

Distinct unmatched codes: 682
Unmatched accidents: 88281
Share: 1.85 %


**4. Rates** 
* Accidents per 1,000 inhabitants and per km², example using the 2024

In [8]:
df["accidents_per_1000"] = df["accidents"] / df["Popolazione residente"] * 1000
df["accidents_per_km2"] = df["accidents"] / df["Superficie (Kmq)"]

print("Rows with rate:", df["accidents_per_1000"].notna().sum())      
print("Rows without rate:", df["accidents_per_1000"].isna().sum())     
df[["REF_AREA", "TIME_PERIOD", "accidents", "accidents_per_1000", "accidents_per_km2"]].head(5)

Rows with rate: 184126
Rows without rate: 8615


,REF_AREA,TIME_PERIOD,accidents,accidents_per_1000,accidents_per_km2
0,1001,2001,5,1.934236,0.380335
1,1001,2002,5,1.934236,0.380335
2,1001,2003,4,1.547389,0.304268
3,1001,2004,9,3.481625,0.684603
4,1001,2005,2,0.773694,0.152134


**5. Save: all years, all municipalities**

In [9]:
df = df.drop(columns=["Codice Comune (numerico)"])
int_cols = ["Codice Regione", "Capoluogo di Provincia/Uts", "Capoluogo di Regione", "Popolazione residente"]
df[int_cols] = df[int_cols].fillna(0).astype(int)
os.makedirs("Data/Processed", exist_ok=True)
df.to_csv("Data/Processed/integrated_clean.csv", index=False)
print(df.shape)       
df.head(3)

(192741, 14)


,REF_AREA,TIME_PERIOD,accidents,deaths,injured,Comune,Sigla automobilistica,Codice Regione,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione residente,Superficie (Kmq),accidents_per_1000,accidents_per_km2
0,1001,2001,5,0,10,Agliè,TO,1,0,0,2585,13.1463,1.934236,0.380335
1,1001,2002,5,0,10,Agliè,TO,1,0,0,2585,13.1463,1.934236,0.380335
2,1001,2003,4,0,7,Agliè,TO,1,0,0,2585,13.1463,1.547389,0.304268
